In [ ]:
import os
import logging
import chromadb
import re
import pathlib
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1. Configure Production Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
# FIX1
def _extract_field(text: str, field_name: str) -> str | None:
    """
    Case-insensitive field extractor with leading-whitespace tolerance.
    Logs a DEBUG entry when found, returns None silently when absent
    (callers are responsible for logging fallback warnings).
    """
    pattern = re.compile(
        rf"^\s*{re.escape(field_name)}\s*:\s*(.+)$",
        re.IGNORECASE | re.MULTILINE
    )
    match = pattern.search(text)
    if match:
        value = match.group(1).strip()
        return value if value else None
    return None

class ThreatIntelDB:
    def __init__(self, db_path: str = "./brain"):
        """Initializes the persistent vector database."""
        try:
            self.client = chromadb.PersistentClient(path=db_path)
            embedding_fn = SentenceTransformerEmbeddingFunction(
                model_name="all-MiniLM-L6-v2"
            )

            self.collection = self.client.get_or_create_collection(
                name="mitre_threat_intel",
                embedding_function=embedding_fn,  
                metadata={"hnsw:space": "cosine"} 
            )
            # Initialize the splitter for large documents (CISA reports, etc.)
            self.splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                separators=["\n\n", "\n", " ", ""]
            )
            logger.info(f"Successfully connected to ChromaDB at {db_path}")
        except Exception as e:
            logger.error(f"Failed to initialize ChromaDB: {e}")
            raise
    # PRODUCTION VERSION — delete old chunks for this file before upserting new ones
    def _delete_existing_chunks(self, filename: str):
        """Removes all previously ingested chunks for a given source file."""
        try:
            existing = self.collection.get(where={"source": filename})
            if existing and existing["ids"]:
                self.collection.delete(ids=existing["ids"])
                logger.info(f"[{filename}] Deleted {len(existing['ids'])} stale chunks "
                            f"before re-ingestion.")
        except Exception as e:
            logger.warning(f"[{filename}] Could not clean up old chunks: {e}")

    def process_directory(self, directory_path: str):
        """Iterates through a directory, chunks text, and prepares for ingestion."""
        dir_path = pathlib.Path(directory_path)
        if not dir_path.exists() or not dir_path.is_dir():
            logger.error(f"Path '{directory_path}' does not exist or is not a directory.")
            return
            
        txt_files = sorted(dir_path.glob("*.txt"))  # FIX5: deterministic order, no hidden files

        if not txt_files:
            logger.warning(f"No .txt files found in '{directory_path}'. Nothing to ingest.")
            return

        logger.info(f"Found {len(txt_files)} .txt files to process in '{directory_path}'.")

        for filepath in txt_files:
            filename = filepath.name
            if filename.endswith(".txt"):          # ← gate check FIRST
                filepath = os.path.join(dir_path, filename)
                try:                               # ← try sits INSIDE the if
                    with open(filepath, 'r', encoding='utf-8') as f:
                        raw_text = f.read().strip()
                        if not raw_text:  # FIX2
                            logger.warning(f"[{filename}] File is empty — skipping ingestion.")
                            continue  # move to next file in the loop

                        if len(raw_text) < 50:  # arbitrary minimum — a real threat report is never 10 chars
                            logger.warning(f"[{filename}] File suspiciously short ({len(raw_text)} chars) "
                                           f"— ingesting anyway but verify content.")

                    # In process_directory — warn on every fallback, don't silently accept "unknown"
                    technique_id = (
                        _extract_field(raw_text, "TECHNIQUE_ID") or
                        _extract_field(raw_text, "VULNERABILITY_ID")
                    )
                    if not technique_id:
                        technique_id = filename.replace(".txt", "")
                        logger.warning(f"[{filename}] No TECHNIQUE_ID or VULNERABILITY_ID found — "
                                       f"falling back to filename: '{technique_id}'")

                    tactic = _extract_field(raw_text, "TACTIC")
                    if not tactic:
                        tactic = "unknown"
                        logger.warning(f"[{filename}] No TACTIC field found — defaulting to 'unknown'")

                    date_added = _extract_field(raw_text, "DATE_ADDED")
                    if not date_added:
                        date_added = "unknown"
                        logger.warning(f"[{filename}] No DATE_ADDED field found — defaulting to 'unknown'")

                    chunks = self.splitter.create_documents(
                        texts=[raw_text],
                        metadatas=[{
                            "source":       filename,
                            "type":         "threat_intel_report",
                            "technique_id": technique_id,
                            "tactic":       tactic,
                            "date_added":   date_added
                        }]
                    )

                    documents = [c.page_content for c in chunks]
                    metadatas = [c.metadata for c in chunks]
                    ids       = [f"{filename}_{i}" for i in range(len(chunks))]

                    # FIX3: In process_directory, before self.collection.upsert(...)
                    self._delete_existing_chunks(filename)
                    self.collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
                except Exception as e:
                    logger.error(f"Error processing {filename}: {e}")  # ← filename, not filepath
    
    # FIX4
    def semantic_search(self, query: str, n_results: int = 3):
        """
        Returns query results dict on success, empty result structure on no-match,
        or None on hard failure. Caller can distinguish all three cases.
        """
        if not query or not query.strip():
            logger.warning("semantic_search called with empty query — returning None.")
            return None

        if n_results < 1:
            logger.error(f"n_results must be >= 1, got {n_results}.")
            return None

        collection_count = self.collection.count()
        if collection_count == 0:
            logger.warning("semantic_search called on empty collection. "
                       "Run process_directory() first.")
            return None

        # Clamp n_results to what actually exists — prevents ChromaDB crash
        n_results = min(n_results, collection_count)

        try:
            results = self.collection.query(
                query_texts=[query.strip()],
                n_results=n_results
            )
            logger.info(f"Search returned {len(results['documents'][0])} results "
                    f"for query: '{query[:80]}'")
            return results
        except Exception as e:
            logger.error(f"Search failed for query '{query[:80]}': {e}", exc_info=True)
            return None

# --- Execution Block ---
if __name__ == "__main__":
    db = ThreatIntelDB()

    intel_dir = "./threat_reports"

    if not os.path.exists(intel_dir):
        os.makedirs(intel_dir)
        with open(f"{intel_dir}/sample_mitre.txt", "w") as f:
            f.write(
                "TECHNIQUE_ID: T1566\n"
                "TACTIC: Initial Access\n"
                "DATE_ADDED: 2024-01-15\n\n"
                "T1566 - Phishing\n\n"
                "Adversaries may send phishing messages to gain access to victim systems. "
                "This technique involves malicious links or attachments delivered via email. "
                "Defenders should monitor for suspicious email attachments and outbound connections "
                "to newly registered domains following email delivery events.\n"
        )


    db.process_directory(intel_dir)

    question = "How do hackers trick employees into clicking bad links to get inside the network?"
    search_results = db.semantic_search(question)

    if search_results and search_results['documents'][0]:
        logger.info("--- RAG RETRIEVAL RESULTS ---")
        for i in range(len(search_results['documents'][0])):
            doc  = search_results['documents'][0][i]
            meta = search_results['metadatas'][0][i]
            logger.info(f"Source: {meta['source']}")
            logger.info(f"Technique ID: {meta.get('technique_id', 'N/A')}")
            logger.info(f"Tactic: {meta.get('tactic', 'N/A')}")
            logger.info(f"Content: {doc[:300]}...")
    else:
        logger.warning("No results found. Check if documents were ingested correctly.")

In [ ]:
import os
import logging
from dotenv import load_dotenv
import google.generativeai as genai

# 1. Configure Production Logging
# We format the log to include timestamps and severity levels for the SOC audit trail.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

# 2. Secure Initialization
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    logger.error("CRITICAL FAILURE: GEMINI_API_KEY environment variable is missing.")
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")

genai.configure(api_key=api_key)

# 3. Define the Model with Guardrails and Strict Determinism
system_instruction = (
    "You are a Senior SOC Assistant. Use ONLY the provided threat intelligence context. "
    "If the answer isn't explicitly there, you must state: 'I do not have sufficient information."
)

# Force the model to be deterministic (Temperature = 0.0)
safety_config = genai.types.GenerationConfig(
    temperature=0.0, 
    top_p=0.95,      # Standard secondary sampling limit
    top_k=1          # Strictly pick the absolute highest probability token
)

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction=system_instruction,
    generation_config=safety_config
)

def generate_answer(query, search_results):
    """
    Final Generation step with Triage Logic and Audit Logging.
    """
    logger.info(f"Incoming user query: '{query}'")
    
    context_chunks = search_results.get('documents', [[]])[0]

    if not context_chunks:
        logger.warning(f"RAG BYPASS: No relevant threat intel found for query '{query}'. Aborting LLM call to save compute.")
        return "No relevant threat intel found in the database. I cannot answer this query."

    logger.info(f"RAG SUCCESS: Retrieved {len(context_chunks)} chunks. Initiating deterministic generation.")
    
    # Wrap context in XML tags to mathematically set boundaries for attention
    context_text = "\n\n".join(context_chunks)
    
    prompt = f"""
    <threat_intelligence>
    {context_text}
    </threat_intelligence>

    <user_query>
    {query}
    </user_query>
    """

    try:
        response = model.generate_content(prompt)
        logger.info("Generation successful.")
        return response.text
    except Exception as e:
        logger.error(f"LLM Generation failed: {e}")
        return "An error occurred during threat analysis."

# --- Mock Execution ---
if __name__ == "__main__":
    # Case A: Irrelevant query (Triage triggers)
    empty_results = {'documents': [[]]} 
    print(generate_answer("How do I bake a cake?", empty_results))

    print("-" * 40)

    # Case B: Relevant query (Deterministic Generation triggers)
    valid_results = {'documents': [["MITRE T1566: Phishing is commonly executed via malicious PDF attachments."]]}
    print(f"\nAssistant Response:\n{generate_answer('How is phishing executed?', valid_results)}")